In [1]:

from datasets import load_dataset


#loading from hugging face
dataset = load_dataset("tattabio/ec_classification")
## making test and train set
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()
##sanity check to make sure dims are the same as on the DGEB paper dimensions
print(train_df.shape)
print(test_df.shape)

(512, 3)
(128, 3)


In [2]:
train_df.head()
## checking what the label and seq loomks like

,Entry,Label,Sequence
0,Q9LQC0,1.14.14.18,MATSRLNASCRFPASRRLDCESYVSLRAKTVTIRYVRTIAAPRRHL...
1,A0AFT8,1.14.14.18,MIIVTNTIKVEKGAAEHVIRQFTGANGDGHPTKDIAEVEGFLGFEL...
2,P74133,1.14.14.18,MTNLAQKLRYGTQQSHTLAENTAYMKCFLKGIVEREPFRQLLANLY...
3,O31534,1.14.14.18,MFVQLRKMTVKEGFADKVIERFSAEGIIEKQEGLIDVTVLEKNVRR...
4,P34697,1.15.1.1,MFMNLLTQVSNAIFPQVEAAQKMSNRAVAVLRGETVTGTIWITQKS...


In [9]:
x_train_seq = train_df["Sequence"].tolist()
y_train_labels = train_df["Label"].tolist()

x_test_seq = test_df["Sequence"].tolist()
y_test_labels = test_df["Label"].tolist()

from sklearn.ensemble import RandomForestClassifier
## defining model
model = RandomForestClassifier()

In [4]:

import dgeb
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
## first input which is just the embeddings i am using dgeb embedding model to make each seq into a vector
embedding_model = dgeb.get_model("facebook/esm2_t6_8M_UR50D")



Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
## converting train data to numerical vectors
x_train_input1 = embedding_model.encode(x_train_seq)[:, -1, :]

/opt/anaconda3/envs/knnlab/lib/python3.10/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
encoding:   0%|          | 0/4 [00:00<?, ?it/s]/opt/anaconda3/envs/knnlab/lib/python3.10/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
encoding: 100%|██████████| 4/4 [04:09<00:00, 62.49s/it]


In [6]:
## doing same for test data
## broke into two cells so we would avoid rerunning

import numpy as np
x_test_p1 = embedding_model.encode(x_test_seq[:127])[:, -1, :]
last_test = embedding_model.encode(x_test_seq[127:])[:, -1, :]
x_test_input1 = np.vstack([x_test_p1, last_test])


In [7]:
from sklearn.metrics import accuracy_score, f1_score 
## checking shape
print(x_train_input1.shape)
print(x_test_input1.shape)
## now fitting model on this info
model.fit(x_train_input1, y_train_labels)
predictions_input1 = model.predict(x_test_input1)

## accuracy and f1 score for input 1
print("accuracy:", accuracy_score(y_test_labels, predictions_input1))
print("f1:", f1_score(y_test_labels, predictions_input1, average="macro"))

(512, 320)
(128, 320)
accuracy: 0.46875
f1: 0.4171875


In [11]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score 
## now doing input 2 which is k mer counts of 3

## defining a function to make kmer
def kmerFunction(seq):
    return " ".join([seq[i:i+3] for i in range(len(seq)-2)])


train_kmer = [kmerFunction(seq) for seq in x_train_seq]
test_kmer = [kmerFunction(seq) for seq in x_test_seq]

## now counting how many times each kmer shows up
helper = CountVectorizer()
x_train_input2 = helper.fit_transform(train_kmer)
x_test_input2 = helper.transform(test_kmer)

## checking shape
print(x_train_input2.shape)
print(x_test_input2.shape)

## defining model again

model2 = RandomForestClassifier()
model2.fit(x_train_input2, y_train_labels)
predictions_input2 = model2.predict(x_test_input2)

## printing accuracy and f1 score
print("accuracy:", accuracy_score(y_test_labels, predictions_input2))
print("f1:", f1_score(y_test_labels, predictions_input2, average="macro"))

(512, 7986)
(128, 7986)
accuracy: 0.0859375
f1: 0.05926339285714285


In [13]:
## now doing input 3 which is weighted kmer
from sklearn.feature_extraction.text import TfidfVectorizer

## defining the weighted helper
weighted =  TfidfVectorizer()

x_train_input3 = weighted.fit_transform(train_kmer)
x_test_input3 = weighted.transform(test_kmer)

## ensuring shape is accurate
print(x_train_input3.shape)
print(x_test_input3.shape)

## making model3
model3 = RandomForestClassifier()

model3.fit(x_train_input3, y_train_labels)
predictions_input3 = model3.predict(x_test_input3)

## f1 and accuracy scores
print("accuracy:", accuracy_score(y_test_labels, predictions_input3))
print("f1:", f1_score(y_test_labels, predictions_input3, average="macro"))

(512, 7986)
(128, 7986)
accuracy: 0.1171875
f1: 0.07994791666666666
